# 06 — Retrieval Evaluation

*Level 2 — Advanced RAG*

## Objective
The capstone comparison: Recall@K, MRR, and NDCG@K for every retrieval strategy built in this level, on the **full 300-query** scifact test set with real qrels — direct evidence for *how much*, if at all, each layer of sophistication actually helped.

`dense`-only is the direct analog of Level 1's approach (embed + cosine top-K) applied to this harder, real IR benchmark — so this table is also the closest apples-to-apples answer this repo can give to "did Level 2 actually beat Level 1?"


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))
sys.path.insert(0, str(LEVEL_DIR / "hybrid-search"))


In [2]:
from common.dataset import prepare
from retrieval.dense import DenseRetriever
from retrieval.sparse import BM25Retriever
from bm25_vector import HybridRetriever
from reranking.cross_encoder import CrossEncoderReranker
from evaluation.recall_at_k import recall_at_k
from evaluation.mrr import mrr
from evaluation.ndcg import ndcg_at_k

data = prepare()
corpus_texts = {d: data.corpus_text(d) for d in data.doc_ids()}
dense = DenseRetriever.from_corpus(corpus_texts)
sparse = BM25Retriever.from_corpus(corpus_texts)
hybrid = HybridRetriever(dense, sparse)
reranker = CrossEncoderReranker()
print(f"Evaluating on {len(data.queries)} queries against real qrels.")


/Users/yessinezghal/Desktop/learn/rag/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 16940.49it/s]

Evaluating on 300 queries against real qrels.


In [3]:
MAX_K = 20

def run_all(queries):
    dense_r, sparse_r, hybrid_r, reranked_r = {}, {}, {}, {}
    for qid, q in queries.items():
        dense_r[qid] = [d for d, _ in dense.search(q, top_k=MAX_K)]
        sparse_r[qid] = [d for d, _ in sparse.search(q, top_k=MAX_K)]
        hybrid_candidates = hybrid.search(q, top_k=MAX_K)
        hybrid_r[qid] = [d for d, _ in hybrid_candidates]
        cand_texts = [(d, corpus_texts[d]) for d, _ in hybrid_candidates]
        reranked_r[qid] = [d for d, _ in reranker.rerank(q, cand_texts, top_k=MAX_K)]
    return dense_r, sparse_r, hybrid_r, reranked_r

dense_results, sparse_results, hybrid_results, reranked_results = run_all(data.queries)
print("done")


done


## Final comparison table


In [4]:
methods = {
    "sparse (BM25)": sparse_results,
    "dense (~ Level 1 style)": dense_results,
    "hybrid (dense+BM25, RRF)": hybrid_results,
    "hybrid + reranked": reranked_results,
}

print(f"{'method':<26}{'Recall@5':>10}{'Recall@10':>11}{'MRR':>8}{'NDCG@10':>10}")
for name, results in methods.items():
    r5 = recall_at_k(results, data.qrels, 5)
    r10 = recall_at_k(results, data.qrels, 10)
    m = mrr(results, data.qrels, k=10)
    n10 = ndcg_at_k(results, data.qrels, 10)
    print(f"{name:<26}{r5:>10.3f}{r10:>11.3f}{m:>8.3f}{n10:>10.3f}")


method                      Recall@5  Recall@10     MRR   NDCG@10
sparse (BM25)                  0.847      0.867   0.751     0.770
dense (~ Level 1 style)        0.900      0.917   0.806     0.831
hybrid (dense+BM25, RRF)       0.887      0.923   0.802     0.825
hybrid + reranked              0.890      0.923   0.788     0.813


## What I observed

The measured table (your numbers may vary slightly run to run — Ollama embeddings are not perfectly deterministic across versions/hardware, but the pattern is stable):

| method | Recall@5 | Recall@10 | MRR | NDCG@10 |
|---|---|---|---|---|
| sparse (BM25) | 0.847 | 0.867 | 0.751 | 0.770 |
| dense (~ Level 1 style) | 0.900 | 0.917 | **0.806** | **0.831** |
| hybrid (dense+BM25, RRF) | 0.887 | **0.923** | 0.802 | 0.825 |
| hybrid + reranked | 0.890 | **0.923** | 0.788 | 0.813 |

The honest, slightly humbling finding: **plain dense retrieval posted the best MRR and NDCG@10 of all four methods.** Hybrid ties it for the best Recall@10, and reranking on top of hybrid didn't improve MRR or NDCG at all here — it *reduced* both compared to hybrid alone, for the same reason found in `04_reranking.ipynb`: a general-domain cross-encoder reordering candidates for a specialized biomedical corpus.

This is not a failure of the techniques — it is exactly the result Level 2 exists to make visible. Every technique in this level is a *tool*, not a *guarantee*:

1. **Chunking** — fixed-size is the only strategy guaranteed to cut through a sentence; recursive/semantic/parent-child exist to avoid that at some added complexity.
2. **Dense beat sparse** on this corpus because scifact's queries paraphrase rather than reuse terminology — the opposite would likely hold on a corpus full of exact identifiers.
3. **Hybrid roughly tied dense-only** — fusing a strong and a weaker ranker doesn't guarantee a net win on every metric.
4. **Reranking measurably underperformed hybrid-only on MRR/NDCG here** — an off-the-shelf, general-domain cross-encoder is not automatically better on a specialized (biomedical) corpus.
5. **Query transformations are a targeted fix, not a blanket upgrade** — HyDE solved a query in `05_query_transformations.ipynb` that every plain method missed, but cost an extra LLM call to do it.

The only way to know which combination is actually better for *your* corpus is to measure it, exactly like this notebook does — not to assume the more sophisticated pipeline automatically wins.

## Next

[Level 3 — Modular RAG](../../03-modular-rag/README.md) — once a single retrieval pipeline is measurably tuned, the next question is *which* retrieval pipeline a given query should even go to.
